In [12]:
import os
import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()
API_KEY = os.getenv("API_FOOTBALL_KEY")

if API_KEY:
    print("API key detectee.")
else:
    print("API key manquante: ajoute API_FOOTBALL_KEY dans .env")

API key detectee.


In [13]:
from datetime import datetime, timezone

FINISHED_STATUSES = {"FT", "AET", "PEN"}
FIXTURE_COLUMNS = [
    "Match_ID",
    "Date",
    "Timestamp",
    "Status",
    "League",
    "LeagueId",
    "Season",
    "Home",
    "Away",
    "HomeTeamId",
    "AwayTeamId",
    "Venue",
    "HomeGoals",
    "AwayGoals",
    "HomeGoalsRaw",
    "AwayGoalsRaw",
    "HT_Home",
    "HT_Away",
    "IsFinished",
]

def fetch_fixtures(params: dict, api_key: str) -> list[dict]:
    if not api_key:
        raise ValueError("API_FOOTBALL_KEY est manquante. Ajoute-la dans .env.")

    url = "https://v3.football.api-sports.io/fixtures"
    headers = {"x-apisports-key": api_key}

    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()
    payload = response.json()
    return payload.get("response", [])

def fixtures_to_dataframe(fixtures: list[dict]) -> pd.DataFrame:
    rows = []
    for item in fixtures:
        fixture = item.get("fixture", {})
        league = item.get("league", {})
        teams = item.get("teams", {})
        goals = item.get("goals", {})
        score = item.get("score", {})
        venue = fixture.get("venue") or {}

        home_goals = goals.get("home")
        away_goals = goals.get("away")
        status = (fixture.get("status") or {}).get("short")

        rows.append(
            {
                "Match_ID": fixture.get("id"),
                "Date": fixture.get("date"),
                "Timestamp": fixture.get("timestamp"),
                "Status": status,
                "League": league.get("name"),
                "LeagueId": league.get("id"),
                "Season": league.get("season"),
                "Home": (teams.get("home") or {}).get("name"),
                "Away": (teams.get("away") or {}).get("name"),
                "HomeTeamId": (teams.get("home") or {}).get("id"),
                "AwayTeamId": (teams.get("away") or {}).get("id"),
                "Venue": venue.get("name"),
                "HomeGoals": home_goals if home_goals is not None else 0,
                "AwayGoals": away_goals if away_goals is not None else 0,
                "HomeGoalsRaw": home_goals,
                "AwayGoalsRaw": away_goals,
                "HT_Home": (score.get("halftime") or {}).get("home"),
                "HT_Away": (score.get("halftime") or {}).get("away"),
                "IsFinished": status in FINISHED_STATUSES,
            }
        )

    frame = pd.DataFrame(rows, columns=FIXTURE_COLUMNS)
    if not frame.empty:
        frame["Date"] = pd.to_datetime(frame["Date"], utc=True, errors="coerce")

    return frame

def fetch_match_data(date: str, api_key: str) -> pd.DataFrame:
    return fixtures_to_dataframe(fetch_fixtures({"date": date}, api_key))

def fetch_team_recent_matches(team_id: int, api_key: str, last_matches: int = 8) -> pd.DataFrame:
    params = {
        "team": team_id,
        "last": last_matches,
        "status": "FT",
    }
    return fixtures_to_dataframe(fetch_fixtures(params, api_key))

today_utc = datetime.now(timezone.utc).strftime("%Y-%m-%d")
df_all = fetch_match_data(today_utc, API_KEY)

# Garde uniquement les matchs de la competition "World Cup"
df = df_all[df_all["League"].fillna("").str.casefold().eq("world cup")].reset_index(drop=True)

print(f"Matchs World Cup recuperes pour {today_utc}: {len(df)} / {len(df_all)}")
df[["Date", "Status", "Home", "Away", "Venue"]].head(20)

Matchs World Cup recuperes pour 2026-06-15: 4 / 71


,Date,Status,Home,Away,Venue
0,2026-06-15 02:00:00+00:00,FT,Sweden,Tunisia,Estadio BBVA
1,2026-06-15 16:00:00+00:00,NS,Spain,Cape Verde Islands,Mercedes-Benz Stadium
2,2026-06-15 19:00:00+00:00,NS,Belgium,Egypt,Lumen Field
3,2026-06-15 22:00:00+00:00,NS,Saudi Arabia,Uruguay,Hard Rock Stadium


In [14]:
import pandas as pd
# Assurez-vous que la fonction calculate_elo est disponible ou réimportez le fichier elo_calculator
from elo_calculator import calculate_elo 

ELO_INITIAL = 1500 
K_FACTOR = 32 # Maintenu pour référence

def poisson_predict(lambda_home: float, lambda_away: float) -> tuple[int, int]:
    """Simule la prédiction du score basé sur deux moyennes de buts attendus."""
    print(f"Simulation : Home est attendu à {lambda_home:.2f} but(s), Away à {lambda_away:.2f} but(s).")
    # On retourne les valeurs entières les plus proches des moyennes.
    return round(max(0, lambda_home)), round(max(0, lambda_away))

def calculate_expected_goals(row) -> tuple[float, float]:
    """Estime les taux moyens attendus de buts (lambda) basés sur le différentiel ELO."""
    # Ceci est la formule mathématique ajustée : 1 + diff / C
    home_rating = row['HomeRating']
    away_rating = row['AwayRating']

    # Constantes: Baseline de 1 but, Ajustement de sensibilité (900.0)
    lambda_h = max(0.8, 1 + (home_rating - away_rating) / 900.0)  # For Home Expected Goals
    lambda_a = max(0.8, 1 + (away_rating - home_rating) / 900.0) # For Away Expected Goals

    return lambda_h, lambda_a

In [15]:
def predict_scores(df: pd.DataFrame, api_key: str) -> pd.DataFrame:
    """
    Orchestre la prédiction de scores : calcule les forces (ELO), 
    estime les buts attendus ($\lambda$), et prédit le score basé sur Poisson.
    """
    print("--- Étape 1 : Calcul de la force des équipes (Indice Elo) ---")
    # 1. Calcul du rating ELO basé sur tous l'historique
    _, elo_ratings = calculate_elo(df)
    print(f"Ratings calculés pour {len(elo_ratings)} équipes.")
    
    team_ratings_df = pd.DataFrame(list(elo_ratings.items()), columns=['Team', 'ELO'])
    
    # 2. Préparation des features (Jointure de ratings)
    print("\n--- Étape 2 : Préparation pour la Modélisation de Score ---")
    
    df_featured = df.merge(team_ratings_df, left_on='Home', right_on='Team', how='left').rename(columns={'ELO': 'HomeRating'})
    df_featured = df_featured.merge(team_ratings_df, left_on='Away', right_on='Team', how='left').rename(columns={'ELO': 'AwayRating'})

    # On garde uniquement les colonnes nécessaires pour la prédiction
    prediction_data = df_featured[[
        'Home', 'Away', 'HomeGoals', 'AwayGoals', 
        'HomeRating', 'AwayRating'
    ]].dropna()

    print(f"Données prêtes pour la modélisation statistique : {len(prediction_data)} matchs.")

    # 3. Implémentation du Modèle Statistique (Distribution de Poisson) - Prédiction des buts attendus ($\lambda$)
    print("\n--- Étape 3 : Simulation des Taux Moyens Attendus (Lambda) ---")
    
    # Application de la fonction pour obtenir les paires (\lambda_h, \lambda_a)
    prediction_data[['Lambda_Home', 'Lambda_Away']] = prediction_data.apply(
        lambda_results = [calculate_expected_goals(row) for _, row in prediction_data.iterrows()], axis=1
    )
    
    # 4. Génération des scores prédits (Expected Value)
    prediction_data[['Predicted_Home', 'Predicted_Away']] = pd.DataFrame(
        [poisson_predict(lambda_h, lambda_a) for lambda_h, lambda_a in zip(prediction_data['Lambda_Home'], prediction_data['Lambda_Away'])], 
        index=prediction_data.index
    )

    return prediction_data[['Home', 'Away', 'Predicted_Home', 'Predicted_Away']]

<>:4: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
<>:4: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
C:\Users\Shadow\AppData\Local\Temp\ipykernel_6700\901994687.py:4: SyntaxWarning: "\l" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\l"? A raw string is also an option.
  estime les buts attendus ($\lambda$), et prédit le score basé sur Poisson.


In [16]:
if 'df' in globals():  # Vérifie si `df` existe dans l'environnement du notebook
    print("--- Démarrage de la Prédiction Complète ---")
    final_predictions = predict_scores(df, API_KEY)
    
    print("\n===============================================")
    print("⭐ PRÉDICTIONS FINALES (Home vs Away) ⭐")
    print("===============================================")
    display(final_predictions.head())
else:
    print("ERREUR : Le DataFrame 'df' contenant les matchs World Cup n'est pas trouvé dans l'environnement.")


--- Démarrage de la Prédiction Complète ---
--- Étape 1 : Calcul de la force des équipes (Indice Elo) ---


NameError: name 'std_dev' is not defined